# Notebook 2 — Exploratory Data Analysis

**Goals**

1. Compute extended descriptive statistics — moments, dispersion, share
   of zeros — for both numeric and categorical columns.
2. Visualise distributions and time series with informative plots.
3. Test for stationarity using the Augmented Dickey-Fuller test.
4. Read autocorrelation (ACF) and partial autocorrelation (PACF) plots.
5. Decompose a series into trend, seasonality and residuals.
6. Spot calendar effects (day-of-week, month-of-year).


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))

import forecasting_toolkit as ft
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────

DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_subset.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount','holiday_name',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 352,834 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,9,2020-08-01,Prague_1,4111.0,21.15,27.45,0.42,0.0,0.0,0.0,...,8,2020,2020,1,True,False,3,1,213,Czechia
1,9,2020-08-02,Prague_1,4804.0,7.94,27.45,0.61,0.0,0.0,0.0,...,8,2020,2020,2,False,False,3,1,214,Czechia
2,9,2020-08-03,Prague_1,5012.0,12.41,27.45,0.87,0.0,0.0,0.0,...,8,2020,2020,3,False,False,3,0,215,Czechia
3,9,2020-08-04,Prague_1,4832.0,21.71,27.45,1.00,0.0,0.0,0.0,...,8,2020,2020,4,False,False,3,0,216,Czechia
4,9,2020-08-05,Prague_1,4843.0,28.49,27.45,1.00,0.0,0.0,0.0,...,8,2020,2020,5,False,False,3,0,217,Czechia


## 2.1 Descriptive statistics — numeric columns

`describe_numeric` is `df.describe()` plus what forecasting actually
cares about: **skewness**, **kurtosis**, **coefficient of variation**, and
**share of zeros**.

- *Skewness* > 1 → consider a log or Box-Cox transform later.
- *Kurtosis* much greater than 3 → heavy tails, expect outliers.
- *CV* > ~1 → highly variable; harder to forecast.
- High *zero_pct* on the target → intermittent demand (next notebook).


In [3]:
ft.eda.describe_numeric(df)


,column,count,mean,std,min,p25,median,p75,max,iqr,skewness,kurtosis,cv,zero_pct
0,unique_id,352834,2644.139343,1579.317446,9.000000,1283.000000,2608.000000,4053.00000,5402.000000,2770.000000,0.046036,-1.226416,0.597290,0.00
1,total_orders,352834,6033.824672,2475.795183,458.000000,4455.000000,5600.000000,8030.00000,18475.000000,3575.000000,0.210882,-0.383704,0.410319,0.00
2,sales,352834,105.247111,227.240935,0.000000,18.350000,39.980000,96.38000,8418.390000,78.030000,6.632967,72.659100,2.159118,0.88
3,sell_price_main,352834,174.510250,383.454902,0.370000,20.750000,44.930000,118.22000,3629.320000,97.470000,4.226616,21.422349,2.197320,0.00
4,availability,352834,0.928583,0.176127,0.010000,1.000000,1.000000,1.00000,1.000000,0.000000,-2.882174,8.146409,0.189672,0.00
5,type_0_discount,352834,0.008055,0.072160,-20.949300,0.000000,0.000000,0.00000,0.973450,0.000000,-149.309586,42383.020068,8.958138,96.49
6,type_1_discount,352834,0.000213,0.006348,0.000000,0.000000,0.000000,0.00000,0.207790,0.000000,30.278096,926.009744,29.739502,99.88
7,type_2_discount,352834,0.000926,0.016346,0.000000,0.000000,0.000000,0.00000,0.667110,0.000000,21.340256,523.950393,17.658030,99.59
8,type_3_discount,352834,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,100.00
9,type_4_discount,352834,0.006750,0.031413,0.000000,0.000000,0.000000,0.00000,0.256570,0.000000,4.468503,18.172245,4.653750,95.56


## 2.2 Descriptive statistics — categorical columns

For categorical / object columns the questions change. We care about:

- *cardinality* (`n_unique`) — high-cardinality columns may need
  frequency or target encoding rather than one-hot expansion.
- *mode share* — does one category dominate?


In [4]:
ft.eda.describe_categorical(df)


,column,n_unique,missing,mode,mode_share_pct,top_values
0,warehouse,7,0,Prague_2,18.85,"Prague_2(66526), Brno_1(66438), Prague_1(62416..."
1,name,465,0,Shallot_2,0.80,"Shallot_2(2827), Bread_98(2821), Lemon_16(2819..."
2,L1_category_name_en,3,0,Fruit and vegetable,46.14,"Fruit and vegetable(162790), Bakery(151566), M..."
3,L2_category_name_en,38,0,Fruit and vegetable_L2_3,21.68,"Fruit and vegetable_L2_3(76477), Fruit and veg..."
4,L3_category_name_en,125,0,Bakery_L3_16,5.44,"Bakery_L3_16(19202), Bakery_L3_75(17602), Frui..."
5,L4_category_name_en,47,0,Fruit and vegetable_L4_1,33.05,"Fruit and vegetable_L4_1(116629), Bakery_L4_1(..."
6,holiday_name,36,338461,NaN,95.93,"nan(338461), International womens day(1018), G..."
7,is_month_start,2,0,False,96.73,"False(341285), True(11549)"
8,is_month_end,2,0,False,96.75,"False(341365), True(11469)"
9,country,3,0,Czechia,72.72,"Czechia(256568), Hungary(55720), Germany(40546)"


## 2.3 Distribution of the target

A histogram + boxplot stacked together is the fastest way to see shape,
spread and outliers at once.


In [5]:
fig = ft.plotting.plot_distribution(df, TARGET_COL, bins=60,
                                    title=f'Distribution of {TARGET_COL}')
fig.show()


If your target is heavily right-skewed, a log-y axis often makes the
shape readable:


In [6]:
fig = ft.plotting.plot_distribution(df, TARGET_COL, bins=60, log_y=True,
                                    title=f'Distribution of {TARGET_COL} (log y)')
fig.show()


## 2.4 Time series visualisation

We saw a small selection in Notebook 1. Now let's pick a single series
and look at it in detail — that's where ACF, PACF and decomposition
operate.


In [7]:
keys = ft.data_io.summarize_keys(df, spec)
example_key_dict = keys.iloc[0][KEY_COLS].to_dict()
example_key_dict


{'unique_id': 80}

In [8]:
single = ft.data_io.get_series(df, spec, example_key_dict)
print(f'Single series: {len(single)} rows')

# Index by date so the next plots / decomposition can use a real time index
single_ts = single.set_index(DATE_COL)[TARGET_COL].astype(float)
fig = ft.plotting.plot_time_series(df, spec, keys=[example_key_dict],
                                   title=f'Selected series')
fig.show()


Single series: 1416 rows


## 2.5 Stationarity — the Augmented Dickey-Fuller test

A **stationary** series has constant mean, variance and autocorrelation
structure over time. Many classical models (ARMA, ARIMA before differencing)
assume stationarity. The ADF test is a formal hypothesis test:

- **H₀ (null):** the series has a unit root → *non-stationary*.
- **H₁:** the series is stationary.

Reject H₀ (p < 0.05) → series is stationary at 5%.

>  Statistical significance ≠ economic relevance. ADF rejecting the
> null doesn't mean the series has *no* trend, just that the test couldn't
> find a unit root.


In [9]:
adf = ft.eda.adf_test(single_ts.dropna())
for k, v in adf.items():
    print(f'  {k:25s} {v}')


  test_statistic            -1.406223081396867
  p_value                   0.579289085496176
  used_lag                  13
  n_obs                     1402
  crit_values               {'1%': -3.4350228340280737, '5%': -2.86360372349629, '10%': -2.567868718663576}
  is_stationary_5pct        False


## 2.6 Autocorrelation — ACF and PACF

**Autocorrelation function (ACF)**: correlation of the series with its
own past at lag *k*.

**Partial autocorrelation function (PACF)**: like ACF, but with the
effects of intermediate lags removed.

Pattern → diagnosis cheat-sheet:

| Pattern in ACF                       | Likely cause          |
|--------------------------------------|-----------------------|
| Slow linear decay                    | Trend (non-stationary)|
| Spike at lag 7 (daily data)          | Weekly seasonality    |
| Spike at lag 12 (monthly data)       | Yearly seasonality    |
| Sharp cut-off at lag *p*             | MA(*p*) signal        |
| Tails off                            | AR signal             |

| Pattern in PACF        | Likely cause |
|------------------------|--------------|
| Sharp cut-off at lag p | AR(*p*) signal |
| Tails off              | MA signal      |

The 95% confidence band is shaded — only spikes outside the band are
considered statistically significant.


In [10]:
acf_pacf = ft.eda.compute_acf_pacf(single_ts.dropna(), nlags=40)
fig = ft.plotting.plot_acf_pacf(acf_pacf, title='ACF / PACF — selected series')
fig.show()


## 2.7 Seasonal decomposition

Classical decomposition splits the series into three additive components:

$$y_t = T_t + S_t + R_t$$

where T is the slow-moving trend, S is a repeating seasonal pattern, and
R is the residual. The `period` argument is the length of one season:

| Frequency | Reasonable `period` |
|-----------|---------------------|
| daily, weekly seasonality | 7 |
| daily, yearly seasonality | 365 |
| monthly, yearly seasonality | 12 |
| hourly, daily seasonality | 24 |


In [11]:
# Pick a period that fits your frequency. Defaulting to 7 (weekly) for daily data.
SEASON_PERIOD = 7

#decomp = ft.eda.seasonal_decompose(single_ts.dropna(), period=SEASON_PERIOD,
#                                   model='additive')

decomp = ft.eda.stl_decompose(single_ts.dropna(), period=SEASON_PERIOD)

fig = ft.plotting.plot_decomposition(decomp,
       title=f'Seasonal decomposition (period = {SEASON_PERIOD})')
fig.show()


### Quantifying trend and seasonal strength

We can summarise *how much* of the series is explained by trend vs
seasonality on a 0-1 scale (Wang, Smith & Hyndman 2006). 0 = none,
1 = entirely explained by that component.


In [12]:
strength = ft.eda.seasonal_strength(decomp)
print(f"Trend strength    : {strength['trend_strength']:.3f}")
print(f"Seasonal strength : {strength['seasonal_strength']:.3f}")


Trend strength    : 0.949
Seasonal strength : 0.436


## 2.8 Calendar heatmap — patterns at a glance

A year × month-of-year heatmap shows seasonal effects and year-on-year
changes in one picture. Aggregates across all keys by default — pass
a `key_values` dict to focus on one series.


In [13]:
fig = ft.plotting.plot_calendar_heatmap(df, spec, key_values=example_key_dict, aggregate='mean')
fig.show()


## 2.9 Correlations between numeric columns

If you have **dynamic** features (price, promo flag, weather, etc.), the
correlation matrix is the quickest way to see which ones move with the
target.

>  Use `method='spearman'` instead of `'pearson'` for rank correlation —
> it's robust to outliers and to non-linear monotonic relationships.


In [14]:
numeric_cols = [TARGET_COL] + DYNAMIC_COLS
numeric_cols = [c for c in numeric_cols if c in df.columns]

if len(numeric_cols) >= 2:
    corr = ft.eda.correlation_matrix(df[numeric_cols], method='spearman')
    fig = ft.plotting.plot_correlation_heatmap(corr, title='Spearman correlation')
    fig.show()
else:
    print('Need at least 2 numeric columns to compute a correlation matrix.')
    print('Add columns to DYNAMIC_COLS in the dataset block at the top.')


## 2.10 Take-aways

A useful EDA write-up answers four questions:

1. **What does a typical series look like?** (mean, variance, shape)
2. **Is it stationary?** (ADF test)
3. **What seasonality does it have?** (ACF spikes, decomposition)
4. **Which features look correlated with the target?** (correlation matrix)
